# CSV, Parquet and Partitioned Datasets in Spark

**Dataset:** 2013 New York City flights data (`nycflights13`), distributed as CSV from a public GitHub repository. We will create a Parquet copy from those same real records, so the comparison is apples-to-apples.

## Before you start
This notebook stores working files in a Unity Catalog Volume rather than relying on DBFS root or `/tmp`, which is restricted in serverless environments.

**One-time setup:** in Catalog Explorer, create a volume named `bda_week7` in a schema where you have write permission. The examples below assume:
`/Volumes/workspace/default/bda_week7/`

If your workspace uses a different catalog/schema, change `BASE_PATH` in the next cell. Do not put personal or sensitive data in this teaching volume.

---
## 1. Environment Setup

Spark SQL is the SQL interface to Apache Spark. Under the hood every SQL query is compiled into the same directed acyclic graph (DAG) of operations as a programmatic DataFrame query.

The cells below install – as usual – PySpark and Java (required as Spark runs on the JVM), then start a **`SparkSession`**, the single entry point to all Spark functionality.


In [0]:
#!pip install pyspark "pyarrow>=15.0.0"

# Install Java 17 (required for Spark)
#!sudo apt-get update
#!sudo apt-get install -y openjdk-17-jdk-headless
#!java -version

# Set JAVA_HOME so PySpark can find the JVM
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("CSV_Parquet") \
        .getOrCreate()

A Volume in Databricks is a governed storage location where you can store and access files, such as CSV files, Parquet files, JSON files, images, or other data files.

Think of it as a folder in cloud storage that Databricks manages access to. It lets users and applications access files through a standard path, with permissions controlled by Unity Catalog.

For example, in our practical class:
```
/Volumes/main/default/bda_week7/
    raw/
        flights.csv
    curated/
        flights_parquet/
            part-00000-....parquet
```

- raw/ stores the original CSV dataset.
- curated/ stores the Parquet version created by Spark.
- The Volume provides the storage location for these files.
Why do we use a Volume?

In Databricks, a Volume provides a way to organize and access files while managing who is allowed to read or write them. This is particularly useful when working with cloud storage in a governed environment.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

BASE_PATH = "/Volumes/workspace/default/bda_week7"
RAW_DIR = f"{BASE_PATH}/raw"
CURATED_DIR = f"{BASE_PATH}/curated"
PARTITIONED_DIR = f"{BASE_PATH}/partitioned"
CSV_PATH = f"{RAW_DIR}/flights.csv"
PARQUET_PATH = f"{CURATED_DIR}/flights_parquet"

print("Working directory:", BASE_PATH)

### 1. Get the CSV data

The source is the `flights` table from the public `nycflights13` dataset. It describes flights departing New York City airports in 2013, including carrier, origin, destination, scheduled/actual times, and distance. This cell stores the original CSV into the Volume we just created.

In [0]:
# flights.csv is already in the week07 workspace folder — let's copy it into the Volume.
LOCAL_CSV = "/Workspace/Users/dcaridade@novaims.unl.pt/BigDataAnalysis2627/weeks/week07/flights.csv"

# Create folders in the Volume.
dbutils.fs.mkdirs(RAW_DIR)
dbutils.fs.mkdirs(CURATED_DIR)
dbutils.fs.mkdirs(PARTITIONED_DIR)

# Copy the local CSV into the Volume.
dbutils.fs.cp(LOCAL_CSV, CSV_PATH, True)
print("Copied flights.csv to", CSV_PATH)

# Verify the file is there.
print("Files in raw/:")
dbutils.fs.ls(RAW_DIR)

### 2. Read CSV and inspect it

CSV stores text fields separated by delimiters. The header contains column names, but the file does not carry a reliable typed schema. Here we ask Spark to infer types so we can inspect what it decides. In production pipelines, consider specifying an explicit schema for predictable ingestion.


In [0]:
flights_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(CSV_PATH)
)

print("Rows:", flights_csv.count())
flights_csv.printSchema()
display(flights_csv.limit(10))


### 3. Select only the fields needed for analysis

Analytical queries often need only a small subset of a wide dataset. Parquet's columnar layout allows readers to avoid reading unrelated column chunks. CSV is text and must parse the records to identify fields.

In [0]:
sample_query = (
    flights_csv
    .select("month", "carrier", "origin", "dest", "distance", "air_time")
    .filter(F.col("month") == 7)
)
display(sample_query.limit(10))
sample_query.explain("formatted")

#### Understanding the Physical Plan

The output of `sample_query.explain("formatted")` shows how Spark plans to execute the query. The physical plan contains several operations:

- **(1) Scan CSV:** Spark reads the data from `flights.csv`. The `ReadSchema` shows that Spark reads only the six columns required by the query, rather than all columns in the dataset. This is an example of **column pruning**.

- **`PushedFilters`:** Spark pushes the filter conditions `IsNotNull(month)` and `EqualTo(month,7)` towards the data source. This is called **predicate pushdown**. However, because the source is a CSV file, the presence of pushed filters does not necessarily mean Spark can avoid reading the underlying file's contents.

- **(2) PhotonRowToColumnar:** Converts data from row format to columnar format for processing by Photon.

- **(3) PhotonFilter:** Applies the filter condition, keeping only rows where `month = 7` (July).

- **(4) PhotonProject:** Keeps and arranges the six requested columns in the query output.

- **(5) PhotonColumnarToRow:** Converts the data back to row format when needed to produce the result.

- **(6) PhotonResultStage:** Represents the final stage that produces the query results.

The final message, **“The query is fully supported by Photon,”** means that Photon, Databricks' query execution engine, can execute the entire query.

**Key takeaway:** Spark optimizes the query by reading only the required columns and pushing the month filter towards the data source. However, since the input is CSV, Spark may still need to scan the file to identify the rows that match the filter. With a partitioned Parquet dataset, Spark may be able to skip entire partitions when filtering on a partition column.

## 4. Write a Parquet copy and read it back

Parquet is a typed, columnar format with schema metadata and support for encoding/compression. The Parquet copy is generated from the same real CSV records, so later comparisons use equivalent data.

`mode("overwrite")` replaces the target path. Only use it here because this is a dedicated lab output directory.


In [0]:
# Write a non-partitioned Parquet dataset
(
    flights_csv.write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

flights_parquet = spark.read.parquet(PARQUET_PATH)
print("Parquet rows:", flights_parquet.count())
flights_parquet.printSchema()
display(flights_parquet.limit(10))

### 5. Compare storage footprint

Spark writes a *directory* containing one or more part files, not necessarily one file. The helper below totals the sizes of data files in each directory. File sizes can vary with Spark settings and the number of output tasks, so focus on the observed result rather than assuming a fixed compression ratio.

In [0]:
def directory_bytes(path):
    total = 0
    for item in dbutils.fs.ls(path):
        if item.isDir():
            total += directory_bytes(item.path)
        elif not item.name.startswith("_") and not item.name.startswith("."):
            total += item.size
    return total

csv_bytes = directory_bytes(RAW_DIR)
parquet_bytes = directory_bytes(PARQUET_PATH)
print(f"CSV directory size:     {csv_bytes:,} bytes")
print(f"Parquet directory size: {parquet_bytes:,} bytes")
if csv_bytes:
    print(f"Parquet / CSV size:     {parquet_bytes / csv_bytes:.2%}")

## Exercise 1: 
Run the same query on flights_csv and flights_parquet, selecting the columns ("month", "carrier", "origin", "distance") and filtering where month == 7. Use `.explain("formatted")` to compare the scan formats, column pruning, and pushed filters.

In [0]:
# Using the csv
csv_query = (
    flights_csv
    .select("month", "carrier", "origin", "distance")
    .filter(F.col("month") == 7)
)

print("=== CSV Physical Plan ===")
csv_query.explain("formatted")

In [0]:
# Using the parquet
parquet_query = (
    flights_parquet
    .select("month", "carrier", "origin", "distance")
    .filter(F.col("month") == 7)
)

print("=== Parquet Physical Plan ===")
parquet_query.explain("formatted")

#### Observations:
1. The CSV scan node reports "FileScan csv" while Parquet reports "FileScan parquet".
2. Both plans show column pruning — only 4 columns appear in ReadSchema, not all 19. However, Parquet physically skips reading the unrequested column chunks on disk, while CSV still has to parse every character of every line to locate the fields.
3. PushedFilters: both show IsNotNull(month) and EqualTo(month,7). With Parquet these can be pushed into the columnar reader for row-group skipping; with CSV Spark must parse the entire file regardless.

## Exercise 2:
Run the same aggregation on flights_csv and flights_parquet, calculating the average distance by carrier, rounding to two decimal places, and ordering by carrier. Use `.explain("formatted")` to compare the number of columns scanned, shuffle stages, and any additional overhead in the CSV plan.

In [0]:
# Using the csv
csv_agg = (
    flights_csv
    .groupBy("carrier")
    .agg(F.round(F.avg("distance"), 2).alias("avg_distance"))
    .orderBy("carrier")
)

print("=== CSV Aggregation Results ===")
display(csv_agg)
print("=== CSV Physical Plan ===")
csv_agg.explain("formatted")

In [0]:
# Using the parquet
parquet_agg = (
    flights_parquet
    .groupBy("carrier")
    .agg(F.round(F.avg("distance"), 2).alias("avg_distance"))
    .orderBy("carrier")
)

print("=== Parquet Aggregation Results ===")
display(parquet_agg)
print("=== Parquet Physical Plan ===")
parquet_agg.explain("formatted")

#### Observations:
1. CSV must parse the full text of every row to extract carrier and distance. Parquet's ReadSchema shows only [carrier, distance] — the columnar reader physically reads just those two column chunks from disk.
2. Both plans have the same number of Exchange (shuffle) stages for the groupBy, so the logical plan structure is identical.
3. The CSV plan includes a CSV parsing / deserialisation step that the Parquet plan avoids entirely — this is where Parquet saves CPU on large datasets.

## Exercise 3:
Compare narrow and wide projections on flights_parquet and flights_csv, filtering where distance > 1000. Use `.explain("formatted")` to compare column pruning and explain why Parquet is more efficient for analytical workloads.

In [0]:
# --- CSV: narrow projection (only carrier and distance) ---
csv_narrow = (
    flights_csv
    .select("carrier", "distance")
    .filter(F.col("distance") > 1000)
)
print("=== CSV NARROW ===")
csv_narrow.explain("formatted")

In [0]:
# --- CSV: wide projection (all columns) ---
csv_wide = (
    flights_csv
    .select("*")
    .filter(F.col("distance") > 1000)
)
print("=== CSV WIDE ===")
csv_wide.explain("formatted")

In [0]:
# --- Parquet: narrow projection (only carrier and distance) ---
parquet_narrow = (
    flights_parquet
    .select("carrier", "distance")
    .filter(F.col("distance") > 1000)
)
print("=== Parquet NARROW ===")
parquet_narrow.explain("formatted")

In [0]:
# --- Parquet: wide projection (all columns) ---
parquet_wide = (
    flights_parquet
    .select("*")
    .filter(F.col("distance") > 1000)
)
print("=== Parquet WIDE ===")
parquet_wide.explain("formatted")

#### Solving it using spark sql

In [0]:
flights_csv.createOrReplaceTempView("flights")

csv_narrow = spark.sql(
    '''
    SELECT carrier, distance
    FROM flights
    WHERE distance > 1000
    '''
)

csv_narrow.explain("formatted")

In [0]:
csv_wide = spark.sql(
    '''
    SELECT *
    FROM flights
    WHERE distance > 1000
    '''
)

csv_wide.explain("formatted")

In [0]:
flights_parquet.createOrReplaceTempView("flights2")

parquet_narrow = spark.sql(
    '''
    SELECT carrier, distance 
    FROM flights2
    WHERE distance > 1000
    '''
)

parquet_narrow.explain("formatted")

In [0]:
parquet_wide = spark.sql(
    '''
    SELECT * 
    FROM flights2
    WHERE distance > 1000
    '''
)

parquet_wide.explain("formatted")

#### Observations:
1. Parquet NARROW ReadSchema shows only [carrier, distance]; Parquet WIDE shows all 19 columns. The columnar format physically skips reading the unrequested column chunks.
2. CSV NARROW and CSV WIDE may both list only the needed columns in Spark's ReadSchema, but the underlying CSV reader still parses every character of every line to locate the fields — there is no byte-level column skipping like Parquet has.
3. This is why Parquet is preferred for analytical workloads: column pruning reduces disk I/O, and predicate pushdown into row groups can skip entire blocks without scanning them at all.

### 6. Prepare useful partition columns

For this dataset, `month` and `origin` are plausible partition columns for a teaching example: analysts may filter by month or airport, and the number of combinations is manageable. This is not a universal recommendation as partitioning should reflect common query filters and data volume.

We will write by `origin` then `month`. The resulting folders encode values, such as `origin=JFK/month=7/`.


In [0]:
# Use the Parquet DataFrame as the input for the partitioned output
partition_input = flights_parquet

(
    partition_input.write
    .mode("overwrite")
    .partitionBy("origin", "month")
    .parquet(PARTITIONED_DIR)
)

print("Wrote partitioned Parquet to:", PARTITIONED_DIR)

### 7. Inspect the physical directory layout

A partitioned dataset is a directory tree. The partition values are stored in folder names; each folder contains Parquet part files. Do not expect exactly one file per folder: the number depends on the write and execution layout.

In [0]:
display(dbutils.fs.ls(PARTITIONED_DIR))

In [0]:
def show_tree(path, depth=2, indent=""):
    if depth < 0:
        return
    for item in dbutils.fs.ls(path):
        print(indent + item.name + ("/" if item.isDir() else f" ({item.size:,} bytes)"))
        if item.isDir() and depth > 0:
            show_tree(item.path, depth - 1, indent + "  ")

show_tree(PARTITIONED_DIR, depth=1)

### 8. Read the dataset root and query partition columns

Read the **root** directory. Spark discovers `origin` and `month` from the directory names and exposes them as DataFrame columns. Avoid reading a single `part-....parquet` file when you intend to query the whole dataset.

In [0]:
flights_partitioned = spark.read.parquet(PARTITIONED_DIR)
flights_partitioned.printSchema()
display(flights_partitioned.select("origin", "month", "carrier", "distance").limit(10))

### 9. Query a specific partition

The query below filters on both partition columns. Spark can use **partition pruning** to avoid considering folders that cannot match the filter. It still applies row-level filtering where needed; pruning does not replace filtering.

In [0]:
jfk_july = (
    flights_partitioned
    .filter((F.col("origin") == "JFK") & (F.col("month") == 7))
    .select("carrier", "dest", "distance", "air_time")
)

display(jfk_july.limit(20))
jfk_july.explain("formatted")

### 10. Compare a partition-column filter with a non-partition filter

The first query filters on `origin` and `month`, which are partition columns. The second filters on `dest`, which is not a partition column in this layout. Spark can still filter on `dest`, but it cannot prune directories based on `dest` values from this folder layout.


In [0]:
by_partition = flights_partitioned.filter(
    (F.col("origin") == "JFK") & (F.col("month") == 7)
)
by_destination = flights_partitioned.filter(F.col("dest") == "SFO")

print("Plan: partition-column filter")
by_partition.explain("formatted")

print("Plan: non-partition-column filter")
by_destination.explain("formatted")

%md
The partition-column filter uses `PartitionFilters` to select only the partitions where `origin = JFK` and `month = 7`, allowing Spark to skip irrelevant partition directories. In contrast, the destination filter appears under `RequiredDataFilters` and `DictionaryFilters`, since `dest` is not a partition column and Spark must scan the relevant data files to find matching rows.

### Exercise 4:

1. Write a new partitioned Parquet dataset using `month` first and `origin` second.
2. Read the root directory.
3. Return the count of flights from LaGuardia (`LGA`) in December.
4. Inspect the plan. What is different from the original layout, and what remains the same?

Use a new output path so you do not overwrite the earlier example.

In [0]:
ALT_PARTITIONED_DIR = f"{BASE_PATH}/partitioned_month_origin"

# 1. Write a new partitioned Parquet dataset using month first, origin second.
(
    flights_parquet.write
    .mode("overwrite")
    .partitionBy("month", "origin")
    .parquet(ALT_PARTITIONED_DIR)
)
print("Wrote partitioned Parquet to:", ALT_PARTITIONED_DIR)

# 2. Read the root directory.
flights_partitioned_alt = spark.read.parquet(ALT_PARTITIONED_DIR)

# 3. Count flights from LaGuardia (LGA) in December.
lga_dec_count = (
    flights_partitioned_alt
    .filter((F.col("origin") == "LGA") & (F.col("month") == 12))
    .count()
)
print(f"Flights from LGA in December: {lga_dec_count:,}")

# 4. Inspect the plan.
lga_dec = (
    flights_partitioned_alt
    .filter((F.col("origin") == "LGA") & (F.col("month") == 12))
)
print("\n=== Physical Plan ===")
lga_dec.explain("formatted")

#### What changed vs the original (origin, month) layout:
- The directory tree is now month=N/origin=X instead of origin=X/month=N.
- Partition pruning still works — Spark pushes both filters into PartitionFilters and skips all directories that don't match month=12 and origin=LGA.

#### What stays the same:
- The number of partitions is identical (same distinct month×origin combinations).
- The row data inside each Parquet file is unchanged; only the folder structure differs.
- Both partition columns are still available as DataFrame columns after reading the root.

### Exercise 5:
Compare the two partition layouts by filtering on `month == 7` only. Run the same query on both `flights_partitioned` (partitioned by `origin`, `month`) and `flights_partitioned_alt` (partitioned by `month`, `origin`). Use `.explain("formatted")` to compare the `PartitionFilters` and the number of partition directories scanned. Which layout prunes more directories when filtering on `month` alone, and why does partition column order matter?


In [0]:
# Filter on the original layout (origin, month)
month_filter_original = (
    flights_partitioned
    .filter(F.col("month") == 7)
)

print("=== Original layout (origin, month) — filter on month ===")
month_filter_original.explain("formatted")

In [0]:
# Filter on the reversed layout (month, origin) — month is the FIRST level
month_filter_reversed = (
    flights_partitioned_alt
    .filter(F.col("month") == 7)
)

print("=== Reversed layout (month, origin) — filter on month ===")
month_filter_reversed.explain("formatted")

#### Observations:
1. In the reversed layout (month, origin), month is the top-level partition column. Spark prunes immediately at the root — it opens only the month=7/ directory and skips the other 11 month folders entirely. PartitionFilters shows (month = 7) as a direct top-level prune.
2. In the original layout (origin, month), month is nested under origin. Spark must list all 3 origin= directories (JFK, EWR, LGA) and then look inside each for a month=7/ subfolder. The same PartitionFilter (month = 7) is applied, but Spark touches more directories to discover which ones match.
3. Partition column order matters because Spark prunes top-down through the directory tree. Placing the most frequently filtered column first means fewer directory listings and less metadata overhead. This is a small gain here (3 origins), but on datasets with hundreds of top-level partitions the difference is significant.